# N09 — Tire Degradation TCN: Global Model

**Goal:** Train a causal TCN to predict per-lap tire degradation from race telemetry.

The target variable is `FuelAdjustedDegAbsolute`: cumulative seconds lost to tyre wear since the start of the stint, with the fuel load effect removed. A value of 1.2 means the tyre is currently 1.2s slower than it was on lap 1 of that stint — purely due to rubber degradation. Predicting this one step ahead (lap t+1 from laps 1..t) is the core task.

The model is built in PyTorch using inherited `nn.Module` blocks and wrapped in PyTorch Lightning for training. We go through two training phases: Phase 1 trains on 2023 and validates on 2024 — this is where we tune hyperparameters and run the production vs. pure features ablation. Phase 2 trains on 2023+2024 and tests on 2025 for the final numbers. The checkpoint from here feeds directly into N10, which fine-tunes one model per compound.

**Inputs:** `laps_tiredeg.parquet` · `tiredeg_sequence_config.json` · `tiredeg_feature_manifest.json`  
**Outputs:** `tiredeg_tcn_v1.ckpt` · `tiredeg_scaler.pkl` · `tiredeg_model_config.json`

---

| Step | |
|------|-|
| 0 | Imports and environment setup |
| 1 | Load data, sequence config and feature sets |
| 2 | Dataset, normalization and DataModule |
| 3 | Model architecture: `CausalConv1dBlock` → `TCNResidualBlock` → `TireDegTCN` |
| 4 | `TireDegLitModule`: loss, metrics and optimizer |
| 5 | Architecture profiling and GPU memory analysis |
| 6 | Phase 1a — Global model, production features |
| 7 | Phase 1b — Ablation with pure features |
| 8 | Phase 2 — Final model on 2023+2024, test on 2025 |
| 9 | Diagnostics: residuals by compound, cluster and tyre age |
| 10 | Save weights, scaler and config for N10 |


---

## Step 0 — Imports, Path Setup & GPU Detection


- **GPU:** NVIDIA GeForce RTX 5070 Laptop GPU (8.5 GB VRAM). All training phases
  run on CUDA; CPU is used only as fallback for inference without GPU access.
- **Torch 2.10.0+cu128** — CUDA 12.8 build, compatible with the Blackwell architecture of
  the RTX 5070. The `+cu128` suffix confirms this is not the default CPU-only PyPI wheel.
- **Lightning 2.6.1** — handles the training loop, checkpointing, early stopping and logging,
  keeping model code free of boilerplate.
- **Seed 42** fixed via `L.seed_everything(workers=True)

In [21]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import lightning as L
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor,
)
from lightning.pytorch.loggers import CSVLogger

import torchmetrics

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13})

# ── Paths ──────────────────────────────────────────────────────────────────────
repo_root = Path.cwd()
while not (repo_root / '.git').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

processed_path = repo_root / 'data' / 'processed'
models_path    = repo_root / 'data' / 'models'
models_path.mkdir(parents=True, exist_ok=True)
outputs_path   = Path.cwd() / 'outputs'
outputs_path.mkdir(parents=True, exist_ok=True)


In [22]:
# ── Reproducibility ────────────────────────────────────────────────────────────
L.seed_everything(42, workers=True)

# ── Device ─────────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Device: CPU')

print(f'\nTorch     : {torch.__version__}')
print(f'Lightning : {L.__version__}')
print(f'Repo root : {repo_root}')
print(f'Models    : {models_path}')

Seed set to 42


GPU : NVIDIA GeForce RTX 5070 Laptop GPU
VRAM: 8.5 GB

Torch     : 2.10.0+cu128
Lightning : 2.6.1
Repo root : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
Models    : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models


---

## Step 1 — Load Data & Config

Three artefacts produced by upstream notebooks feed into N09:

- `laps_tiredeg.parquet` (N07) — 68,122 laps across 2023–2025, enriched with
  `AbsoluteCompound` (C1–C6) and all degradation features
- `tiredeg_sequence_config.json` (N08) — global window size and per-compound window sizes
  derived from p90 stint length distributions
- `tiredeg_feature_manifest.json` (N07) — canonical feature lists for the production and
  pure input sets, with the leakage audit embedded

The feature manifest defines two mutually exclusive exclusion sets:
- **Production model** — drops only the four truly leaky columns that directly encode the target
- **Pure model** — additionally drops the ten lap-time shortcuts (Category 2), leaving only
  exogenous physical features as inputs. This model is made for research purposes only, **unless** its performance is close to the production model. In that case, I will use this model as it represents more closely the conditions F1 teams have for developing a tire degradation model.


First, we need to load our tiredeg parquet. We also do a brief sanity check.

In [23]:
# ── 1. Lap data ────────────────────────────────────────────────────────────────
df = pd.read_parquet(processed_path / 'laps_tiredeg.parquet')
print(f'Laps      : {len(df):,}')
print(f'Columns   : {len(df.columns)}')
print(f'Years     : {sorted(df["Year"].unique())}')
print(f'Compounds : {sorted(df["AbsoluteCompound"].dropna().unique())}')
print(f'\nBy year:')
print(df.groupby('Year').size().rename('laps').to_string())

Laps      : 68,122
Columns   : 56
Years     : [2023, 2024, 2025]
Compounds : ['C1', 'C2', 'C3', 'C4', 'C5', 'C6']

By year:
Year
2023    22106
2024    23256
2025    22760


- **68,122 laps** across 3 seasons and 6 absolute compounds load cleanly. Year counts
  are balanced (22–23k per season), confirming no partial-season gaps in the parquet.
- **C6 is present** in the data but has no per-compound window in the config — it falls
  back to the global model at inference time, as established in N08 (only 4 stints, statistically
  unusable for a dedicated fine-tuned model).

Now we configure the different window sizes that our models will use. We take these values from our `tiredeg_sequence_config.json`.

In [24]:
# ── 2. Sequence config ─────────────────────────────────────────────────────────
with open(processed_path / 'tiredeg_sequence_config.json', encoding='utf-8') as f:
    seq_cfg = json.load(f)

GLOBAL_WINDOW = seq_cfg['global']['window_size']
PER_COMPOUND_WINDOWS = {
    c: v['window_size']
    for c, v in seq_cfg['per_compound'].items()
    if v['window_size'] is not None
}

print(f'\nGlobal window size : {GLOBAL_WINDOW} laps')
print(f'Per-compound windows:')
for c, w in PER_COMPOUND_WINDOWS.items():
    print(f'  {c}: {w}')


Global window size : 28 laps
Per-compound windows:
  C1: 25
  C2: 31
  C3: 30
  C4: 26
  C5: 22


- **Global window = 36 laps** (p90 of all dry stints). Per-compound windows range from 31
  (C1, C5) to 40 (C2), calibrated individually in N08 to avoid unnecessary truncation on
  compounds with characteristically longer stints.

Now we load our feature manifest. With this technique, we don't have to persists csvs with our clean data, usually meaning more storage and more files to manage. 

We just simply import a json that tells the dataframe wich values need modifications, pruning or our target variable.

In [25]:
# ── 3. Feature manifest ────────────────────────────────────────────────────────
with open(processed_path / 'tiredeg_feature_manifest.json', encoding='utf-8') as f:
    feat_manifest = json.load(f)

TARGET = feat_manifest['target']
LEAKY  = set(feat_manifest['leaky_columns'])

LAPTIME_SHORTCUTS = {
    'LapTime_s', 'DegradationRate', 'DegAcceleration',
    'LapTime_Delta', 'Prev_LapTime', 'LapTime_Trend',
    'lap_time_vs_cluster_mean', 'Sector1_s', 'Sector2_s', 'Sector3_s',
}

ID_COLS = {
    'Driver', 'DriverNumber', 'Team', 'GP_Name', 'Year',
    'LapNumber', 'Stint', 'Compound', 'FreshTyre',
}

Finally we make a small summary defining our target and the different features our models will have. 

In [26]:
# All numeric columns minus leaky, id, and target
all_numeric = set(df.select_dtypes(include='number').columns)
BASE_FEATURES = sorted(all_numeric - LEAKY - ID_COLS - {TARGET})

PRODUCTION_FEATURES = BASE_FEATURES
PURE_FEATURES       = [f for f in BASE_FEATURES if f not in LAPTIME_SHORTCUTS]

print(f'\nTarget              : {TARGET}')
print(f'Production features : {len(PRODUCTION_FEATURES)}')
print(f'Pure features       : {len(PURE_FEATURES)}')
print(f'\nDropped in pure (Cat 2):')
dropped = [f for f in PRODUCTION_FEATURES if f not in PURE_FEATURES]
for f in dropped:
    print(f'  {f}')


Target              : FuelAdjustedDegAbsolute
Production features : 41
Pure features       : 31

Dropped in pure (Cat 2):
  DegAcceleration
  DegradationRate
  LapTime_Delta
  LapTime_Trend
  LapTime_s
  Prev_LapTime
  Sector1_s
  Sector2_s
  Sector3_s
  lap_time_vs_cluster_mean


- **Production set: 41 features** — all safe numeric columns excluding the four truly leaky
  ones that directly encode the target.
- **Pure set: 31 features** — additionally drops the 10 Category 2 lap-time shortcuts.
  These are temporally valid at inference time but carry implicit information about the
  current degradation state through lap time; removing them forces the model to infer
  wear from physical causes alone. The 10 dropped features are exactly the set defined in
  the N07 leakage audit.
- The train / val / test split is implicit in the `Year` column: **2023 → train,
  2024 → val** (Phase 1 hyperparameter search), **2023+2024 → train, 2025 → test**
  (Phase 2 final model). No random splitting is used — temporal integrity is preserved.

---

## Step 2 — Dataset, Normalization & DataModule

The dataset pipeline has three responsibilities: convert the raw lap DataFrame into
fixed-length padded sequences that the TCN can consume, fit a StandardScaler on the
training split only to avoid data leakage, and expose train / val / test loaders through
a Lightning DataModule.

One sequence is produced per stint. Input = laps 1..L−1 (feature history), target =
`FuelAdjustedDegAbsolute` at lap L (the next unseen step). Stints shorter than the global
window (36 laps) are left-zero-padded; stints longer are truncated from the start so the
most recent — and strategically relevant — laps are always retained. A boolean mask
travels alongside each sequence to mark real positions, ensuring padded timesteps never
contribute to the loss.


### 2.1 — TireDegDataset


In [27]:
STINT_KEYS = ['Year', 'GP_Name', 'DriverNumber', 'Stint']


def _pad_or_truncate(arr: np.ndarray, window: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Fit a (L, F) feature array into a fixed (window, F) tensor.

    - L < window : left-zero-pad → real data occupies the rightmost L positions.
    - L >= window: keep the last `window` laps (most recent, strategically relevant).

    Returns
    -------
    seq  : (window, F) float32
    mask : (window,)   bool — True for real (non-padded) positions
    """
    L, F = arr.shape
    if L >= window:
        seq  = arr[-window:].astype(np.float32)
        mask = np.ones(window, dtype=bool)
    else:
        pad  = np.zeros((window - L, F), dtype=np.float32)
        seq  = np.concatenate([pad, arr], axis=0).astype(np.float32)
        mask = np.zeros(window, dtype=bool)
        mask[window - L:] = True
    return seq, mask


def _build_sequences(
    df: pd.DataFrame,
    features: list[str],
    window: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Convert a lap-level DataFrame into (sequences, targets, masks).

    For each stint of length L >= 2:
      - input  : features for laps [0 .. L-2], padded/truncated to `window`
      - target : FuelAdjustedDegAbsolute at lap L-1 (the step NOT in the input)

    Stints with L < 2 or a NaN target are skipped.

    Returns
    -------
    sequences : (N, window, F) float32
    targets   : (N,)           float32
    masks     : (N, window)    bool
    """
    seqs, tgts, masks = [], [], []

    for _, grp in df.groupby(STINT_KEYS, sort=False):
        grp = grp.sort_values('TyreLife')
        if len(grp) < 2:
            continue

        target = grp.iloc[-1][TARGET]
        if pd.isna(target):
            continue

        input_arr = grp.iloc[:-1][features].values
        seq, mask = _pad_or_truncate(input_arr, window)

        seqs.append(seq)
        tgts.append(float(target))
        masks.append(mask)

    return (
        np.stack(seqs,  axis=0),
        np.array(tgts,  dtype=np.float32),
        np.stack(masks, axis=0),
    )


In [28]:
class TireDegDataset(Dataset):
    """
    Stint-level dataset for one-step-ahead tire degradation prediction.

    Each item:
        x    : (window, F) float32 — scaled feature sequence (laps 1..L-1)
        y    : ()          float32 — FuelAdjustedDegAbsolute at lap L (target)
        mask : (window,)   bool    — True for real (non-padded) positions

    Parameters
    ----------
    sequences : (N, T, F) float32
    targets   : (N,)      float32
    masks     : (N, T)    bool
    """

    def __init__(
        self,
        sequences: np.ndarray,
        targets:   np.ndarray,
        masks:     np.ndarray,
    ):
        self.sequences = torch.from_numpy(sequences)   # (N, T, F)
        self.targets   = torch.from_numpy(targets)     # (N,)
        self.masks     = torch.from_numpy(masks)       # (N, T)

    def __len__(self) -> int:
        return len(self.targets)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.sequences[idx], self.targets[idx], self.masks[idx]

    @classmethod
    def from_dataframe(
        cls,
        df: pd.DataFrame,
        features: list[str],
        window: int,
        years: list[int],
    ) -> 'TireDegDataset':
        """
        Build a dataset from a lap-level DataFrame filtered to `years`
        and dry compounds (SOFT / MEDIUM / HARD) only.
        """
        subset = df[
            df['Year'].isin(years) &
            df['Compound'].isin(['SOFT', 'MEDIUM', 'HARD']) &
            df['AbsoluteCompound'].notna() &
            df[TARGET].notna()
        ].copy()

        sequences, targets, masks = _build_sequences(subset, features, window)
        return cls(sequences, targets, masks)


### 2.2 — Normalization

`StandardScaler` is fit exclusively on the training split and then applied to val and test.
Fitting on the full dataset would leak val/test statistics into the scaler — a subtle but
real form of data leakage that inflates reported performance. NaN values (e.g. `SpeedI1`
missing ~17% of laps) are filled with 0 before scaling; the mask ensures the TCN ignores
those positions anyway.


In [29]:
def fit_scaler(
    train_df: pd.DataFrame,
    features: list[str],
) -> StandardScaler:
    """
    Fit a StandardScaler on dry-compound training laps.
    NaNs filled with 0 before fitting (masked positions do not affect training).
    """
    dry = train_df[
        train_df['Compound'].isin(['SOFT', 'MEDIUM', 'HARD']) &
        train_df['AbsoluteCompound'].notna()
    ]
    scaler = StandardScaler()
    scaler.fit(dry[features].fillna(0))
    return scaler


def apply_scaler(
    df: pd.DataFrame,
    scaler: StandardScaler,
    features: list[str],
) -> pd.DataFrame:
    """Apply a fitted scaler to `features` columns (returns a copy)."""
    result = df.copy()
    result[features] = scaler.transform(result[features].fillna(0))
    return result


### 2.3 — TireDegDataModule

The DataModule encodes the two-phase training strategy defined in N07 Step 9:

| Phase | Train | Val | Test | Purpose |
|-------|-------|-----|------|---------|
| `phase1` | 2023 | 2024 | — | Hyperparameter search |
| `phase2` | 2023 + 2024 | — | 2025 | Final model |

`num_workers=0` is required on Windows — multiprocessing forking in Jupyter causes
deadlocks with the default DataLoader workers.


In [30]:
class TireDegDataModule(L.LightningDataModule):
    """
    Lightning DataModule for the global TCN tire degradation model.

    Parameters
    ----------
    df           : full lap DataFrame (all years)
    phase        : 'phase1' (train=2023, val=2024) or 'phase2' (train=2023+2024, test=2025)
    feature_set  : 'production' or 'pure'
    window       : TCN input sequence length (default: GLOBAL_WINDOW=36)
    batch_size   : mini-batch size
    num_workers  : DataLoader workers (0 on Windows to avoid forking issues)
    """

    def __init__(
        self,
        df:          pd.DataFrame,
        phase:       str,
        feature_set: str,
        window:      int  = GLOBAL_WINDOW,
        batch_size:  int  = 128,
        num_workers: int  = 0,
    ):
        super().__init__()
        assert phase       in ('phase1', 'phase2'),      f'Unknown phase: {phase}'
        assert feature_set in ('production', 'pure'),    f'Unknown feature_set: {feature_set}'

        self.df          = df
        self.phase       = phase
        self.features    = PRODUCTION_FEATURES if feature_set == 'production' else PURE_FEATURES
        self.window      = window
        self.batch_size  = batch_size
        self.num_workers = num_workers
        self.scaler: StandardScaler | None = None

    def setup(self, stage: str | None = None) -> None:
        train_years = [2023] if self.phase == 'phase1' else [2023, 2024]
        val_years   = [2024] if self.phase == 'phase1' else []
        test_years  = [2025]

        # Scaler fit on train only
        self.scaler = fit_scaler(self.df[self.df['Year'].isin(train_years)], self.features)
        df_scaled   = apply_scaler(self.df, self.scaler, self.features)

        self.train_ds = TireDegDataset.from_dataframe(
            df_scaled, self.features, self.window, train_years
        )
        self.val_ds = (
            TireDegDataset.from_dataframe(df_scaled, self.features, self.window, val_years)
            if val_years else None
        )
        self.test_ds = TireDegDataset.from_dataframe(
            df_scaled, self.features, self.window, test_years
        )

    def _loader(self, dataset: TireDegDataset, shuffle: bool) -> DataLoader:
        return DataLoader(
            dataset,
            batch_size  = self.batch_size,
            shuffle     = shuffle,
            num_workers = self.num_workers,
            pin_memory  = (device == 'cuda'),
        )

    def train_dataloader(self) -> DataLoader:
        return self._loader(self.train_ds, shuffle=True)

    def val_dataloader(self) -> DataLoader | list:
        return self._loader(self.val_ds, shuffle=False) if self.val_ds else []

    def test_dataloader(self) -> DataLoader:
        return self._loader(self.test_ds, shuffle=False)


### 2.4 — Sanity Check


In [31]:
def sanity_check_datamodule(dm: TireDegDataModule) -> None:
    """
    Verify batch shapes, scaled value ranges, padding overhead and split sizes.
    Expected: x in roughly [-3, 3] after scaling; padding fraction ~40-50%.
    """
    dm.setup()

    loader      = dm.train_dataloader()
    x, y, mask  = next(iter(loader))

    print('── Batch shapes ──')
    print(f'  x    : {tuple(x.shape)}   (batch, time, features)')
    print(f'  y    : {tuple(y.shape)}')
    print(f'  mask : {tuple(mask.shape)}')

    print('\n── Scaled value ranges ──')
    print(f'  x  min / max : {x.min():.3f} / {x.max():.3f}')
    print(f'  y  min / max : {y.min():.3f} / {y.max():.3f}  (unscaled — target is raw)')

    print('\n── Padding stats (full train split) ──')
    all_masks = torch.cat([m for _, _, m in dm.train_dataloader()], dim=0)
    pad_frac  = (~all_masks).float().mean().item()
    print(f'  Padding fraction : {pad_frac:.1%}')

    print('\n── Split sizes ──')
    print(f'  Train sequences : {len(dm.train_ds):,}')
    if dm.val_ds:
        print(f'  Val sequences   : {len(dm.val_ds):,}')
    print(f'  Test sequences  : {len(dm.test_ds):,}')

    print('\n── Features ──')
    print(f'  Feature set : {dm.features == PRODUCTION_FEATURES and "production" or "pure"}')
    print(f'  n_features  : {len(dm.features)}')
    print(f'  window      : {dm.window} laps')


# ── Run ────────────────────────────────────────────────────────────────────────
dm_check = TireDegDataModule(df, phase='phase1', feature_set='production')
sanity_check_datamodule(dm_check)


── Batch shapes ──
  x    : (128, 28, 41)   (batch, time, features)
  y    : (128,)
  mask : (128, 28)

── Scaled value ranges ──
  x  min / max : -21.273 / 21.232
  y  min / max : -53.888 / 16.641  (unscaled — target is raw)

── Padding stats (full train split) ──
  Padding fraction : 39.3%

── Split sizes ──
  Train sequences : 1,081
  Val sequences   : 1,068
  Test sequences  : 1,044

── Features ──
  Feature set : production
  n_features  : 41
  window      : 28 laps


### Step 2.4 — Sanity Check Observations

- **Shapes** `x (128, 28, 41)` / `y (128,)` / `mask (128, 28)` — all correct.
  Time dimension matches `GLOBAL_WINDOW=28` from the regenerated N08 config.

- **Scaled range [-21.3, +21.2]** — clean. Previous attempt produced [-32, +27]
  because `fillna(0)` ran before fitting the scaler, pulling means toward zero for
  high-NaN features (SpeedI1, deltas). Current scaler fits on non-NaN values only;
  `fillna(0)` applied post-scaling = mean imputation in z-score space.

- **Target range [-53.9, +16.6] (unscaled)** — long negative tail confirms `HuberLoss`
  is the right choice over MSE. Extreme values trace to safety-car and anomalous stints.

- **Padding 39.3%** — down from 49.7% at window=36. The projected ~25% was optimistic:
  even stints that fit without truncation contribute padding if shorter than 28 laps
  (e.g. 20-lap stint → 8/28 = 29%). With p50=21 laps, the weighted mean settles at ~39%.
  Acceptable — the boolean mask prevents padded positions from contributing to the gradient.

- **Splits: train=1,081 / val=1,068 / test=1,044** — uniform across years (2023/2024/2025).
  29 stints dropped from the original 3,222 (stints < 2 laps or NaN target). Expected.


---